# Notebook 03: Tissue thickness mapping

**Paper section:** Results — The cranial neural tube maintains a dorso-ventral thickness gradient  
**Paper figures:** Fig. 5

This notebook quantifies local tissue thickness by measuring the Euclidean distance from each vertex on the **lumen (inner) surface** mesh to the nearest face of the **basal (outer) surface** mesh.

The resulting vertex-wise scalar field is stored directly on the lumen mesh for integration with curvature data and the spatchcocking projection.

In [ ]:
from vedo import settings
settings.default_backend = "vtk"

import spatchcocking as sp
import numpy as np
import matplotlib.pyplot as plt
from vedo import Mesh

## Load lumen and basal meshes

In [ ]:
lumen_mesh = sp.get_mesh("../data/meshes/HH17/HH17_embryo1_lumen.ply")

# The basal (outer) surface mesh must be generated separately
# from the neuroepithelium segmentation (same protocol as lumen)
basal_mesh = sp.get_mesh("../data/meshes/HH17/HH17_embryo1_basal.ply")

## Compute thickness

For each vertex on the lumen mesh, we compute the distance to the nearest polygon of the basal mesh using vedo's `distance_to` function.

In [ ]:
# distance_to computes point-to-mesh distance and stores it as a vertex array
lumen_mesh = lumen_mesh.distance_to(basal_mesh, signed=False)

thickness = lumen_mesh.pointdata["Distance"]
print(f"Thickness: min={thickness.min():.1f} µm, max={thickness.max():.1f} µm, mean={thickness.mean():.1f} µm")

## Visualize thickness on 3D mesh

In [ ]:
from vedo import Plotter

lumen_mesh.pointdata.select("Distance")
lumen_mesh.cmap("viridis", vmin=0, vmax=thickness.max())

p = Plotter(offscreen=True)
p.show(lumen_mesh, axes=0)
p.screenshot("thickness_3d.png")
from IPython.display import Image
Image("thickness_3d.png")

## Save mesh with thickness data

In [ ]:
# Rename the array for downstream clarity
import numpy as np
lumen_mesh.pointdata["thickness"] = lumen_mesh.pointdata["Distance"]
sp.save_mesh(lumen_mesh, "../data/meshes/HH17/HH17_embryo1_lumen_thickness.ply")

## Thickness distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(thickness, bins=60, color="steelblue", edgecolor="none")
ax.axvline(thickness.mean(), color="red", linestyle="--", label=f"mean = {thickness.mean():.1f} µm")
ax.set_xlabel("Tissue thickness (µm)")
ax.set_ylabel("Vertex count")
ax.set_title("Apico-basal tissue thickness — HH17 embryo 1")
ax.legend()
plt.tight_layout()
plt.show()